# Let's Analyze CrewRift 🔍

**Every game your policy plays leaves a replay. This notebook turns those replays into
decisions about what to build next.**

It is the analysis companion to `LetsPlayCrewrift.ipynb` (build → validate → submit). Work
through it as **scenarios** — the questions you actually ask while developing a policy, each
answered from replay data, each ending with an *artifact you can hand to your coding agent*.

The one fact everything here rests on: **a `.replay` file holds only the players' inputs and a
per-tick state hash.** No positions, no roles, no tasks, no scores. All of that exists only
after the game engine *re-simulates* the inputs — the hash is there to prove the re-sim is
bit-exact. So there are two doors into this notebook:

1. **The bundled data (start here).** 30 real league games, already re-simulated and shipped
   in-repo. Every scenario below runs on it — offline, no account, no game engine.
2. **Your own replays.** Needs the game engine built from source; that is the *Get the engine*
   section near the end, and the capstone walks the full pipeline. (The Nim toolchain takes
   ~20 minutes to set up — if you already know you'll want it, start that section's clone in a
   spare terminal now and keep reading.)


## 1 · Setup

Install the bundled wheels, load the Croatoan nav-mesh, and unpack both data bundles:
`crewrift-trajectory-demo.tgz` (a `swgy-spatial` build: per-tick positions, kills, bodies,
meetings, task completions, and the engine's line-of-sight log) and `crewrift-tasks-demo.tgz`
(the same 30 games as expanded event JSONL — task, vote and chat events).


In [ ]:
# needs network on first run (dependency resolution from PyPI)
%pip install --quiet ./libs/swgy_base-0.3.0-py3-none-any.whl ./libs/swgy_tools-0.5.2-py3-none-any.whl ./libs/swgy_tune-0.0.1-py3-none-any.whl


In [ ]:
%matplotlib inline
import tarfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from swgy_base.nav import load_default_mesh
from swgy_tools.spatial.build import (
    load_bodies, load_heatmaps, load_kills, load_meetings, load_sightings, load_tasks,
)

# Each of swgy_tools' figure helpers selects a headless (Agg) matplotlib backend the first
# time it is imported (they back the file-writing CLIs), which silently disables inline
# figures. Import them all up front so those one-time switches happen here, then re-select
# inline -- every plt.show() below then renders in the notebook (later cached imports of the
# same modules won't flip it back).
import swgy_tools.tasks.render    # noqa: F401
import swgy_tools.route.ribbon    # noqa: F401
import swgy_tools.spatial.render  # noqa: F401
%matplotlib inline

SP = Path("data")            # the swgy-spatial build dir; tools read <SP>/spatial/...
TASKS_JSONL = SP / "tasks"   # the expanded event JSONL, one file per episode
for bundle, marker in [("crewrift-trajectory-demo.tgz", SP / "spatial" / "kills.csv"),
                       ("crewrift-tasks-demo.tgz", TASKS_JSONL)]:
    if not marker.exists():
        with tarfile.open(Path("bundles") / bundle) as t:
            try:
                t.extractall(SP, filter="data")   # py>=3.12: guard against bad paths
            except TypeError:
                t.extractall(SP)

mesh = load_default_mesh()   # the packaged Croatoan nav-mesh (nodes, edges, tasks, vents)
MAP_W, MAP_H = 1235, 659     # world (x, y) == map pixel (x, y), y pointing DOWN

kills = load_kills(SP)
bodies = load_bodies(SP)
tasks = load_tasks(SP)
meetings = load_meetings(SP)
sightings = load_sightings(SP)
by_ep = defaultdict(list)     # sightings indexed by episode -- several scenarios need it
for s in sightings:
    by_ep[s["episode_id"]].append(s)
episodes = sorted(TASKS_JSONL.glob("*.jsonl"))
print(f"{len(episodes)} episodes · {len(kills)} kills · {len(bodies)} bodies · "
      f"{len(tasks)} task completions · {len(meetings)} meetings · "
      f"{len(sightings)} line-of-sight intervals")


In [ ]:
# Who is in these games? Policies are named by their UPLOADER -- treat them as opaque ids.
import json as _json

roster = _json.loads((SP / "spatial" / "roster.json").read_text())   # episode -> {slot: policy}
games = Counter(policy for ep in roster.values() for policy in ep.values())
print("games per policy (this bundle):")
for policy, n in games.most_common():
    print(f"  {n:3d}  {policy}")

# Pick the busiest policy as the stand-in for "my policy" throughout. When you analyze
# your own games (the capstone), set this to your uploader name instead.
MY_POLICY = games.most_common(1)[0][0]
print(f"\nMY_POLICY = {MY_POLICY!r}   (pretend this one is yours)")


In [ ]:
# Cold open, before any method: the single most-witnessed kill in these 30 games, drawn
# from the engine's own line-of-sight log. Blue = the victim's last 30 seconds, amber =
# the killer converging, red x = the kill, aqua = every player who SAW it happen. By the
# end of this notebook you will know how to find this moment in your own games, measure
# how typical it is, and write the policy change that avoids it.
from IPython.display import Image, display

from swgy_tools.spatial.build import episode_tracks
from swgy_tools.spatial.killplot import render_kill, slots_at, witnesses

worst = max(kills, key=lambda k: len(witnesses(by_ep[k["episode_id"]], k)))
tracks, stride = episode_tracks(SP, worst["episode_id"])
png = render_kill(worst, tracks, stride, "data/teaser.png",
                  others=slots_at(SP, worst["episode_id"], worst["tick"]),
                  sightings=by_ep[worst["episode_id"]], dpi=110)
display(Image(filename=str(png)))
n_saw = len(witnesses(by_ep[worst["episode_id"]], worst))
print(f"{worst['killer_policy']} kills {worst['victim_policy']} in the {worst['room']} "
      f"with {n_saw} players watching. Worse: the kill-sites scenario will show this is "
      "not even unusual.")


### Data cheat sheet (for you and your coding agent)

Units first: the engine runs **24 ticks/second**; coordinates are **map pixels**, 1:1 with the
1235×659 Croatoan image, **y pointing down**. Policy names are opaque uploader ids. The killer
on every kill is **authoritative** — the engine stamps who landed it.

| table | one row per | load with | key columns |
|---|---|---|---|
| `positions/<ep>.npz` | player·tick (Playing phase only) | `load_episode_positions` | `tick, slot, x, y, alive, imposter, active_task, vel_x, vel_y, kill_cooldown, vent_cooldown` |
| `kills.csv` | kill | `load_kills` | `tick, killer_slot/policy, victim_slot/policy, x, y, room` |
| `bodies.csv` | corpse | `load_bodies` | `kill_tick, cleared_tick, undiscovered_ticks, reported` |
| `tasks.csv` | completed task | `load_tasks` | `slot, policy, task, tick, room, while_dead` |
| `meetings.csv` | meeting **span** | `load_meetings` | `start, end, kind, timed_out` |
| `visibility.csv` | line-of-sight **interval** | `load_sightings` | `observer_slot, target_slot, tick_start, tick_end` — the engine's own raycast |
| `data/tasks/*.jsonl` | event | `swgy_tools.tasks.eventlog` | votes, chat, manifests, phases — everything the CSVs summarize |

Three semantics worth knowing: `active_task` is the station a player is *working on* right now
(−1 = none) — it separates walking from the 72-tick completion hold. A **meeting teleports every
living player to the Bridge** and no task can progress until play resumes. And **ghosts keep
doing tasks after death** — they walk through walls and nobody can see them, so any per-player
stat must say whether it covers the living, the dead, or both.


## 2 · Scenario: “Is my crew policy wasting time on tasks?”

The reflex when a crew policy under-scores is to optimize its movement — tighter paths, smarter
task order. Before spending a week on that, measure it. Three questions, each against a real
baseline rather than a hunch:

1. **What did it actually do?** — the task swimlane: every completion, in time and on the map.
2. **Was the walking wasteful?** — each task-to-task leg against an **engine-faithful perfect
   follower** (the same physics: acceleration, friction, wall-sliding).
3. **Was the order wasteful?** — the completion order against the **exactly-optimal tour**
   (Held–Karp over the 8 assigned stations; at n=8 exact is cheap, so no approximations).


In [ ]:
# 1. What did MY_POLICY actually do? One lane per game; numbered dots are task completions
# at the tick they happened; hollow rings are GHOST completions (after death); the red x is
# the kill; orange spans are meetings (hatched = the vote ran the clock out).
from IPython.display import Image, display

from swgy_tools.tasks.eventlog import parse_episode
from swgy_tools.tasks.render import render_runs

EPISODES = []
for f in episodes:
    ep = parse_episode(f.read_text().splitlines(), episode_id=f.stem)
    if ep.hash_ok and ep.started:
        EPISODES.append(ep)

runs = [(ep, m) for ep in EPISODES for m in ep.crew if m.name == MY_POLICY]
runs.sort(key=lambda r: r[0].episode_id)
png = render_runs(runs[:10], MY_POLICY, "data/swimlanes.png", dpi=110)
display(Image(filename=str(png)))


In [ ]:
# The strangest economics in the game, computed live from this bundle: dying does not
# stop the work, and meetings stop ALL of it. A murdered crew member keeps completing
# tasks as a ghost -- uninterruptible, unkillable -- while the living lose about half
# the clock to the Bridge.
from swgy_tools.plotstyle import ALIVE, EDGE, GHOST, MEETING, PAGE, SECONDARY

murdered = [m for ep in EPISODES for m in ep.crew if m.death_tick >= 0]
survived = [m for ep in EPISODES for m in ep.crew if m.death_tick < 0]

def done_share(members):
    return 100 * sum(len(m.completions) for m in members) / sum(len(m.assigned) for m in members)

murdered_rate, survivor_rate = done_share(murdered), done_share(survived)
ghost_done = sum(1 for m in murdered for c in m.completions if c.while_dead)
meet_share = 100 * (sum(mt.ticks for ep in EPISODES for mt in ep.meetings)
                    / sum(ep.end_tick for ep in EPISODES))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11.5, 2.9), facecolor=PAGE,
                               gridspec_kw={"width_ratios": [1.0, 1.5]})
for ax in (ax1, ax2):
    ax.set_facecolor(PAGE)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)

ax1.barh([1, 0], [murdered_rate, survivor_rate], height=0.52, color=[GHOST, ALIVE])
ax1.set_yticks([1, 0])
ax1.set_yticklabels([f"murdered crew (n={len(murdered)})", f"survivors (n={len(survived)})"],
                    color=SECONDARY, fontsize=9)
ax1.tick_params(left=False)
for y, v in ((1, murdered_rate), (0, survivor_rate)):
    ax1.text(v + 2, y, f"{v:.0f}%", va="center", color="white", fontsize=11, weight="bold")
ax1.set_xlim(0, 112)
ax1.set_title("share of assignment completed", color="white", fontsize=11, loc="left")

gap = 0.8
ax2.barh([0], [100 - meet_share], height=0.5, color=ALIVE)
ax2.barh([0], [meet_share], left=100 - meet_share + gap, height=0.5, color=MEETING)
ax2.text((100 - meet_share) / 2, 0, f"playing  {100 - meet_share:.0f}%",
         ha="center", va="center", color="white", fontsize=10, weight="bold")
ax2.text(100 - meet_share + gap + meet_share / 2, 0, f"meetings  {meet_share:.0f}%",
         ha="center", va="center", color=EDGE, fontsize=10, weight="bold")
ax2.set_xlim(0, 100 + gap)
ax2.set_ylim(-0.75, 0.75)
ax2.set_title("where the game clock goes (all games, summed)", color="white",
              fontsize=11, loc="left")
plt.tight_layout()
plt.show()
print(f"{ghost_done} of the murdered's completions happened AFTER death -- the ghost shift.")


In [ ]:
# 2 & 3. The two baselines. `swgy-route legs` / `swgy-route tour` are the CLI twins of this.
import statistics as st

from swgy_tools.route.legs import extract_legs
from swgy_tools.route.tour import tours
from swgy_tools.spatial.build import load_episode_positions

task_xy = {t.id: (t.x, t.y) for t in mesh.tasks}
all_eps = sorted({r["episode_id"] for r in tasks})

# Legs: the engine-faithful re-drive is the slow part, so sample 10 of the 30 games here
# and say so on the printout. (`swgy-route legs` runs every game when you want the census.)
LEG_EPS = all_eps[:10]
legs = []
for e in LEG_EPS:
    legs += extract_legs(mesh, e, load_episode_positions(SP, e),
                         load_tasks(SP, e), load_meetings(SP, e), task_xy)
lost = [x.ticks_lost for x in legs if x.ticks_lost is not None]
print(f"{len(legs)} task-to-task legs in {len(LEG_EPS)} of {len(all_eps)} games "
      "(legs spanning a meeting are excluded -- a teleport is not a walk)")
print(f"  ticks lost vs a PERFECT follower : median {st.median(lost):.0f}   mean {st.mean(lost):.0f}")
print(f"  vent hops on crew legs           : {sum(x.vent_hops for x in legs)}  (crew cannot vent; must be 0)")

# Tours: Held-Karp is trivial at n<=8, and the pairwise A* costs are shared across
# crews, so ALL 30 games are scored in ~15s.
ts = tours(mesh, tasks, task_xy)
print(f"\n{len(ts)} crew tours (all {len(all_eps)} games)")
print(f"  actual / optimal walking distance: median {st.median([t.ratio for t in ts]):.2f}x")


### Verdict: don’t tune movement

Those are this bundle's numbers, computed in front of you; the full corpus (952 league games,
measured offline) says the same: a median leg loses **4 ticks** to a perfect follower (6% of
its travel), and the completion order is a median **1.00×** the optimal tour. The whole 8-task
workload is **1,240 ticks of a ~6,000-tick game — meetings eat 4,000**, exactly the split the
time-budget bar above shows. And the ghost economy is the strangest fact in the game:
**murdered crew complete *more* of their assignment than survivors** — a ghost keeps working and can’t be killed again.

Reading the map panel (corpus numbers again): Bridge-cluster stations get done first (mean completion order ~1.6,
98% completed); the far engine rooms come last (~7.2) and are abandoned more than half the
time — not because pathing fails, but because the game ends first. Task optimisation is a
solved problem. Your effort belongs in the scenarios that follow.

**Hand your agent:**

```bash
swgy-task-report data/tasks --glob '*.jsonl' --format csv -o tasks.csv
swgy-route legs data -o legs.csv
swgy-route tour data -o tour.csv
```

> *“tasks.csv is one row per (episode, crew, assigned task) with completion order and tick;
> legs.csv has per-leg `ticks_lost` vs an engine-perfect follower; tour.csv has `ratio` vs the
> optimal task order. Confirm movement isn’t my bottleneck; if any episode disagrees, show me
> which and why.”*


## 3 · Scenario: “Am I voting like the league requires?”

The league’s qualification gate is a **three-skill AND**: before a policy plays ranked games at
all it must vote (`meeting_participation >= 0.5`), hunt (`imposter_kills >= 0.5`), *and* do
tasks (`crew_tasks_mean >= 1.0`). The tasks arm you just measured; the hunting arm is the
kill-sites scenario; this scenario measures the voting arm — the one that fails silently,
because a policy that never votes looks healthy on every other number. And beyond the gate,
voting is where games are actually won: ejecting an imposter is worth more than any task.

The vote events aren’t in the CSV summaries — they’re in the event JSONL. One pass collects
who voted in which meeting, and whether the vote hit an actual imposter.


In [ ]:
# One pass over the event JSONL: manifests (roles), meetings, votes, deaths, outcomes.
# The files are big, so prefilter lines by substring before parsing JSON.
KEYS = ("player_manifest", "vote_called_", "vote_cast", '"kill"', '"died"', "trace_complete")
ROLES, OUTCOMES = {}, {}                    # (ep, slot) -> role ; ep -> outcome
part = defaultdict(lambda: [0, 0])          # policy -> [meetings voted in, meetings alive for]
acc = defaultdict(lambda: [0, 0, 0])        # policy -> [votes at imposters, votes at crew, skips]

for f in episodes:
    ep = f.stem
    names, deaths, meet_ticks, votes = {}, {}, [], []
    for line in f.open():
        if not any(k in line for k in KEYS):
            continue
        r = _json.loads(line)
        k, v, slot = r["key"], r.get("value", {}), r["player"]
        if k == "player_manifest":
            ROLES[(ep, slot)] = v["role"]; names[slot] = v["address"]
        elif k in ("vote_called_body", "vote_called_button"):
            meet_ticks.append(r["ts"])
        elif k == "vote_cast":
            votes.append((slot, r["ts"], v.get("target_slot"), v.get("target") == "skip"))
        elif k == "kill":
            deaths.setdefault(v["victim_slot"], r["ts"])
        elif k == "died":
            deaths.setdefault(slot, r["ts"])
        elif k == "trace_complete":
            OUTCOMES[ep] = v.get("outcome", "unknown")
    imposters = {s for (e, s), role in ROLES.items() if e == ep and role == "imposter"}
    def meeting_of(vote_tick):
        return max((t for t in meet_ticks if t <= vote_tick), default=None)
    for slot, name in names.items():
        alive_meets = [t for t in meet_ticks if t <= deaths.get(slot, 10**9)]
        voted_in = {meeting_of(vt) for (s, vt, _tgt, _sk) in votes if s == slot} - {None}
        part[name][0] += sum(1 for t in alive_meets if t in voted_in)
        part[name][1] += len(alive_meets)
    for slot, _t, tgt, skip in votes:
        name = names.get(slot)
        if name is None:
            continue
        if skip:
            acc[name][2] += 1
        elif tgt in imposters:
            acc[name][0] += 1
        elif tgt is not None:
            acc[name][1] += 1

print(f"{'policy':20} {'participation':>13}  {'gate>=0.5':>9}  {'at imposter':>11} {'at crew':>8} {'skips':>6}")
for name, (v, m) in sorted(part.items(), key=lambda kv: -(kv[1][0] / max(kv[1][1], 1))):
    p = v / max(m, 1)
    hit, miss, sk = acc[name]
    print(f"{name:20} {p:13.2f}  {'PASS' if p >= 0.5 else 'FAIL':>9}  {hit:11d} {miss:8d} {sk:6d}")


### Verdict: check the gate before anything else

If your policy shows up under 0.5 here, nothing else in this notebook matters until it votes.
The second lesson is in the *at imposter / at crew* split: piling onto a forming majority
(Let's Play’s voting brain) participates, but votes that land on crew hand the imposters a free
ejection. A tally that never hits an imposter is a policy voting on noise.

**Hand your agent:** this table (copy the cell output), plus the raw `vote_cast` /
`vote_called_*` rows from any episode’s JSONL.

> *“My participation is X and my votes hit imposters Y% of the time. Given the witnessed-event
> API in swgy_base (`world.witnessed_kill`, `world.suspicious`), propose a vote heuristic that beats tally-following
> without dropping participation below the 0.5 gate.”*


## 4 · Scenario: “Why do I keep dying?”

Dying doesn’t cost you tasks (ghosts finish the list — the tasks scenario showed as much). It costs you the **+100
win**, which dwarfs everything else in the reward. So the question isn’t “how do I do tasks
faster” but “what was true in the moments before I died?” Two views answer it.


In [ ]:
# The kill cam: the victim's and killer's paths over the 30s before one of MY_POLICY's
# deaths, plus every other player and -- from the engine's own line-of-sight log -- who
# could actually SEE the killer do it.
from swgy_tools.spatial.build import episode_tracks
from swgy_tools.spatial.killplot import render_kill, slots_at, witnesses

SELECTED_EPISODE = 2  # Better illustration of the visualization than index zero
my_deaths = [k for k in kills if k["victim_policy"] == MY_POLICY]
kill = my_deaths[SELECTED_EPISODE] if my_deaths else kills[0]
tracks, stride = episode_tracks(SP, kill["episode_id"])
png = render_kill(kill, tracks, stride, "data/killcam.png",
                  others=slots_at(SP, kill["episode_id"], kill["tick"]),
                  sightings=by_ep[kill["episode_id"]], dpi=110)
display(Image(filename=str(png)))
print("blue = victim · amber = killer · red x = the kill · aqua = had line of sight to the killer")


In [ ]:
# The exposure ribbon: the victim's WHOLE game as a line coloured by how many players could
# see them at each tick. Blue stretches = alone (free to work, free to be murdered).
# Only the LIVING trajectory is drawn: a ghost walks through walls and nobody can see it,
# so its ticks would read as "alone" while carrying no risk at all.
from swgy_tools.route.ribbon import render_ribbon
from swgy_tools.spatial.build import load_episode_positions

png = render_ribbon(
    load_episode_positions(SP, kill["episode_id"]), by_ep[kill["episode_id"]],
    kill["victim_slot"], kill["victim_policy"], kill["episode_id"], "data/ribbon.png",
    death_tick=kill["tick"], dpi=110,
)
display(Image(filename=str(png)))


### Verdict: isolation is the risk

The pattern, here and across the 952-game corpus (measured offline): kills land on players in their blue stretches. A crew policy
that hugs well-travelled ground gives an imposter no clean window — and every witnessed kill
feeds the next scenario’s vote. The mesh carries a baked per-node `exposure`/`witnesses`
score (`mesh.exposure_at(x, y)`), so “prefer visible ground” is a one-knob change, cheap
enough to run every tick.

**Hand your agent:** `data/killcam.png` + `data/ribbon.png` (multimodal), and for the numbers
`data/spatial/kills.csv` + `data/spatial/visibility.csv`.

> *“For each of my deaths in kills.csv, compute from visibility.csv how long I had been unseen
> before the kill tick. If the unseen-time distribution is long-tailed, patch my policy to
> bias task choice toward exposed stations (mesh.exposure_at) and re-measure.”*


## 5 · Scenario: “As imposter, where should I kill?”

The imposter half of the same coin. A kill is only free if nobody sees it — and “did anyone
see it” is not a guess here: `visibility.csv` is the engine’s own raycast, interval by
interval. First the league-wide answer, then the map, then the code change.


In [ ]:
# Was the killer SEEN by a third party at the moment of the kill? (Engine line of sight --
# the killer and the victim themselves are excluded.)
wit_room, tot_room = Counter(), Counter()
n_seen = 0
for k in kills:
    tot_room[k["room"]] += 1
    if witnesses(by_ep[k["episode_id"]], k):
        wit_room[k["room"]] += 1
        n_seen += 1

print(f"{len(kills)} kills -- killer seen by a third party in {n_seen} ({100*n_seen/len(kills):.0f}%)\n")
print(f"  {'room the body fell in':22} {'kills':>5}  {'witnessed':>9}")
for room, n in tot_room.most_common():
    print(f"  {room:22} {n:5d}  {100*wit_room[room]/n:8.0f}%")
print("\n(30 games: read the big rooms, not the n=1 tails.)")

# And the worst case, drawn: the cold open from Setup was exactly this kill -- the
# most-witnessed in the bundle. Now the table above says how typical it is.
worst = max(kills, key=lambda k: len(witnesses(by_ep[k["episode_id"]], k)))
tracks, stride = episode_tracks(SP, worst["episode_id"])
png = render_kill(worst, tracks, stride, "data/killcam-worst.png",
                  others=slots_at(SP, worst["episode_id"], worst["tick"]),
                  sightings=by_ep[worst["episode_id"]], dpi=110)
display(Image(filename=str(png)))


In [ ]:
# Where things happen, over the whole bundle. The layer functions are the same ones
# `swgy-spatial-render` uses; composing them yourself is the template for ANY custom
# heatmap your agent writes: bin -> layer -> composite over the basemap. (From-scratch
# first because the recipe is what your agent will reuse; the pre-baked grids in the
# next cell are what you'll load day to day.)
from swgy_tools.plotstyle import PAGE, SECONDARY, TASK_MARK, VENT_MARK, croatoan_axes
from swgy_tools.spatial.render import bodyhide_layer, density_layer, imposter_share_layer

TASK_XY = np.array([(t.x, t.y) for t in mesh.tasks], dtype=int)
VENT_XY = np.array([(v.x, v.y) for v in mesh.vents], dtype=int)

def show_layer(layer, title, cbar_label):
    rgba, norm, cmap = layer
    fig, ax = plt.subplots(figsize=(11.5, 6.4), facecolor=PAGE)
    croatoan_axes(ax, MAP_W, MAP_H)
    ax.imshow(rgba, extent=[0, MAP_W, MAP_H, 0], interpolation="bilinear", zorder=2)
    ax.scatter(TASK_XY[:, 0], TASK_XY[:, 1], marker="s", s=22, facecolors="none",
               edgecolors=TASK_MARK, linewidths=0.7, zorder=4, label="task station")
    ax.scatter(VENT_XY[:, 0], VENT_XY[:, 1], marker="D", s=24, facecolors="none",
               edgecolors=VENT_MARK, linewidths=0.8, zorder=4, label="vent")
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_visible(False)
    ax.set_title(title, color="white", fontsize=12, loc="left", pad=10)
    ax.legend(loc="lower right", frameon=False, fontsize=8, labelcolor=SECONDARY)
    cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
                      fraction=0.028, pad=0.01)
    cb.set_label(cbar_label, color=SECONDARY, fontsize=8)
    cb.outline.set_visible(False)
    plt.show()

# From-scratch example -- occupancy binned with a vectorised scatter-add:
from swgy_tools.spatial.build import load_positions

positions = load_positions(SP)
CELL = 8
gw, gh = (MAP_W + CELL - 1) // CELL, (MAP_H + CELL - 1) // CELL
grid = np.zeros((gh, gw), dtype=np.int64)
live = positions["alive"].astype(bool)
np.add.at(grid, (np.clip(positions["y"][live] // CELL, 0, gh - 1),
                 np.clip(positions["x"][live] // CELL, 0, gw - 1)), 1)
show_layer(density_layer(grid), "player occupancy (built from scratch)", "samples per cell (log)")


In [ ]:
# The pre-baked grids answer the imposter's two placement questions directly.
G, meta = load_heatmaps(SP)

# Where does presence skew imposter? (diverging about the 25% base rate -- 2 of 8 --
# so a cell only reads hot when imposters are OVER-represented, not merely present)
show_layer(imposter_share_layer(G["crew"], G["imposter"]),
           "the imposter tell -- red = disproportionately imposter",
           "imposter share (white = 25% base rate)")

# Where does a corpse stay hidden longest? (a MEAN, so one lucky body doesn't win)
show_layer(bodyhide_layer(G["bodyhide_sum"], G["bodyhide_count"]),
           "where a body stays hidden longest", "mean ticks undiscovered")


In [ ]:
# The offline model against the online outcome. Heat = the mesh's baked per-node exposure
# (`swgy-navmesh-sight` raycasts the wall layer offline -- no gameplay involved); red x =
# where 30 games of real kills landed. If the bake is honest, kills should pool on dark
# (hidden) ground -- and where they don't, the Bridge, the game itself explains it: spawns
# and meetings put everyone there.
from swgy_tools.plotstyle import DEATH

CELL_E = 8
gh, gw = (MAP_H + CELL_E - 1) // CELL_E, (MAP_W + CELL_E - 1) // CELL_E
coords = np.array([(n.x, n.y) for n in mesh.nodes], dtype=float)
expo = np.array([0.5 if n.exposure is None else n.exposure for n in mesh.nodes])
heat = np.zeros((gh, gw))
for gy in range(gh):                    # nearest mesh node per 8px cell, row by row
    cy = gy * CELL_E + CELL_E // 2
    cxs = np.arange(gw) * CELL_E + CELL_E // 2
    d2 = (coords[:, 0][None, :] - cxs[:, None]) ** 2 + (coords[:, 1][None, :] - cy) ** 2
    heat[gy] = expo[np.argmin(d2, axis=1)]
walk8 = mesh.grid.to_mask()[::CELL_E, ::CELL_E][:gh, :gw]   # hide cells with no floor
rgba = plt.get_cmap("inferno")(heat)
rgba[..., 3] = np.where(walk8, 0.55, 0.0)

fig, ax = plt.subplots(figsize=(11.5, 6.4), facecolor=PAGE)
croatoan_axes(ax, MAP_W, MAP_H)
ax.imshow(rgba, extent=[0, MAP_W, MAP_H, 0], interpolation="bilinear", zorder=2)
ax.scatter([k["x"] for k in kills], [k["y"] for k in kills], marker="x", s=46, c=DEATH,
           linewidths=1.7, zorder=5, label=f"kill site ({len(kills)} kills, 30 games)")
ax.set_xticks([])
ax.set_yticks([])
for sp in ax.spines.values():
    sp.set_visible(False)
sm = plt.cm.ScalarMappable(cmap="inferno", norm=plt.Normalize(0, 1))
cb = fig.colorbar(sm, ax=ax, fraction=0.028, pad=0.01)
cb.set_label("baked exposure (0 hidden .. 1 wide open)", color=SECONDARY, fontsize=8)
cb.ax.tick_params(colors=SECONDARY, labelsize=8)
cb.outline.set_visible(False)
ax.set_title("the offline sight-bake vs where kills actually landed", color="white",
             fontsize=12, loc="left", pad=10)
ax.legend(loc="lower right", frameon=False, fontsize=8, labelcolor=SECONDARY)
plt.show()


In [ ]:
# From evidence to code. Weight the mesh's baked exposure by where things actually happen:
def _exposure(x, y):
    e = mesh.exposure_at(int(x), int(y))
    return 0.5 if e is None else e

def weighted_exposure(grid):
    ys, xs = np.nonzero(grid)
    w = grid[ys, xs]
    c = meta["cell"]
    e = np.array([_exposure(x * c + c // 2, y * c + c // 2) for y, x in zip(ys, xs)])
    return float((e * w).sum() / w.sum())

print("exposure, presence-weighted (0 hidden .. 1 wide open), 30 games:")
print(f"  where crew stand      : {weighted_exposure(G['crew']):.2f}")
print(f"  where kills land      : {weighted_exposure(G['kills']):.2f}")
print(f"  where imposters stand : {weighted_exposure(G['imposter']):.2f}")

# The policy change, as a drop-in task chooser. `choose_task` here is Let's Play §3c's
# nearest-task baseline, inlined so this notebook stands alone:
from swgy_base.world import Phase, WorldState

def choose_task(world, me):
    best, best_d2 = None, None
    for k in world.remaining_task_ks:
        rect = world.task_rect(k)
        if rect is None:
            continue
        cx, cy = rect[0] + rect[2] // 2, rect[1] + rect[3] // 2
        d2 = (cx - me[0]) ** 2 + (cy - me[1]) ** 2
        if best_d2 is None or d2 < best_d2:
            best, best_d2 = k, d2
    return best

def choose_task_witnessed(world, me, mesh, safety=200.0):
    """Nearest task, penalized by hiddenness: at safety=200, a fully hidden station must
    be 200px closer than a fully exposed one to win the pick. `safety` is a knob -- give
    it bounds and tune it in Let's Play's knob section."""
    best, best_score = None, None
    for k in world.remaining_task_ks:
        rect = world.task_rect(k)
        if rect is None:
            continue
        cx, cy = rect[0] + rect[2] // 2, rect[1] + rect[3] // 2
        d = ((cx - me[0]) ** 2 + (cy - me[1]) ** 2) ** 0.5
        score = d + safety * (1.0 - _exposure(cx, cy))
        if best_score is None or score < best_score:
            best, best_score = k, score
    return best

demo = WorldState(phase=Phase.PLAYING, map_origin=(0, 0), me_screen=(500, 300),
                  remaining_task_ks=(0, 1),
                  task_markers={0: (448, 154), 1: (648, 380)},   # 0: quiet corner, 1: open floor
                  task_marker_dims={0: (14, 14), 1: (14, 14)})
me = demo.me_collision
print(f"\nnearest-task pick: {choose_task(demo, me)}   "
      f"witnessed pick: {choose_task_witnessed(demo, me, mesh)}  (prefers the open station)")


### Verdict: kill-site choice is the untapped lever

The table above is this bundle's census; the 952-game corpus (measured offline) matches it:
**~48% of kills are witnessed**, and the Bridge — where imposters kill *most* — runs about
**81% witnessed** (it’s the spawn and meeting hub; everyone is standing there), while the
quiet bays run near **0%**. No policy in this bundle appears to gate its kill on “can anyone
see me?” — 30 games is a strong hint rather than a census, but the corpus agrees. And the
data for that check exists live: `world` streams the players you can
see, and the mesh carries baked `exposure`/`witnesses` per node.

The same exposure weighting works for crew (prefer open stations — the dying scenario) and for the
imposter’s inverse (loiter and kill on hidden ground, leave bodies where `bodyhide` runs hot).

**Hand your agent:** the witnessed-by-room table, `data/spatial/heatmaps.npz`, and
`choose_task_witnessed` as the pattern.

> *“Port the witness check to live play: before the imposter presses A, gate on whether any
> visible player has line of sight (world.others + mesh exposure). One bounded knob for the
> gate threshold, tunable offline.”*


## 6 · Get the engine (needed for your own replays)

Everything so far ran on the bundles. To analyze **your own** games you need the game engine —
the analyses require per-tick state, and a `.replay` only *becomes* state when the engine
re-simulates it. The re-simulator is `tools/expand_replay` in the public game repo (the league’s
own qualification gate runs the same tool). From the repo’s README:

```bash
git clone https://github.com/Metta-AI/coworld-crewrift
cd coworld-crewrift
nimby use 2.2.10            # install Nim via nimby: github.com/treeform/nimby
nimby sync -g nimby.lock    # pinned deps (bitworld et al.); also writes nim.cfg
nim c -d:release --opt:speed --hints:off --out:tools/expand_replay tools/expand_replay.nim
./tools/expand_replay | tail -2    # smoke test: no args re-simulates the repo's own
                                   # tests/replays/notsus.bitreplay
export CREWRIFT_EXPAND_REPLAY=$PWD/tools/expand_replay
```

Three things that will bite if unsaid:

- **Version match or nothing.** A replay only re-simulates on the game version that recorded
  it; otherwise the per-tick hash fails (`trace_complete complete:false`; `swgy-spatial` skips
  the episode with an “OUT OF SYNC” warning). Check `swgy-replay-info`’s `game_version` on your
  replays, and build from the matching ref if it disagrees with `master`.
- **The binary embeds its checkout path** (it chdirs there to load the map). Move or delete the
  clone and the binary breaks silently — rebuild after moving.
- **`swgy-replay-info` needs none of this.** Roster, seed, config and duration are in the
  replay header; it’s pure Python. The next cell proves it on the bundled raw replay.


In [ ]:
# No engine needed for this part: the header of a raw .replay is readable in pure Python.
from swgy_tools.replayfile import read_replay

info = read_replay(Path("bundles") / "sample.replay")
print(f"sample.replay: {info.duration_ticks} ticks · seed {info.seed} · "
      f"game_version {info.game_version!r} · {len(info.roster)} players")
for slot in sorted(info.roster):
    print(f"    slot {slot}  {info.roster[slot]}")

# The engine part IS gated: set CREWRIFT_EXPAND_REPLAY (previous cell) to enable the capstone.
import os, subprocess

EXPAND = os.environ.get("CREWRIFT_EXPAND_REPLAY", "")
HAVE_ENGINE = bool(EXPAND) and Path(EXPAND).is_file()
if HAVE_ENGINE:
    out = subprocess.run([EXPAND, "--format", "jsonl", "--snapshot-every", "1000000",
                          str(Path("bundles") / "sample.replay")],
                         capture_output=True, text=True, check=False)
    ok = '"complete":true' in out.stdout
    print(f"\nengine check: {EXPAND}\n  re-simulated sample.replay -> hash "
          f"{'VERIFIED -- you are version-matched' if ok else 'FAILED -- rebuild at the matching ref'}")
    HAVE_ENGINE = ok
else:
    print("\nno engine configured (CREWRIFT_EXPAND_REPLAY unset) -- the capstone cell below "
          "will print instructions instead of running.")


## 7 · Capstone: the full pipeline on YOUR games

With the engine built and a league account — `softmax login` for auth (`uv tool install
softmax-cli` if you don’t have it), and the `coworld` CLI from a clone of
[Metta-AI/coworld](https://github.com/Metta-AI/coworld), run as `uv run coworld` from that
checkout:

```bash
coworld replays --download-dir ./replays --mine     # pull your own league games
```

Then the pipeline is three mechanical steps — re-simulate each replay to event JSONL, build the
spatial dataset, and point every cell above at it:

```bash
for r in replays/*.replay; do
  "$CREWRIFT_EXPAND_REPLAY" --format jsonl --snapshot-every 1 "$r" > "expanded/$(basename $r .replay).jsonl"
done
uv run swgy-spatial expanded/ -o mydata      # positions, kills, visibility, heatmaps...
```

The cell below does exactly that in Python. If `./replays` is empty it uses the bundled
`sample.replay`, so you can prove the pipeline before your first pull. One tip for the agent
loop: `"$CREWRIFT_EXPAND_REPLAY" replays/worst.replay > worst.txt` emits a **plain-text
timeline** of a whole game — the fastest artifact to paste at a coding agent with “find the
first wrong decision.”


In [ ]:
if not HAVE_ENGINE:
    print("Skipping: build the engine (previous section) and re-run. Nothing below needs")
    print("anything else -- the pipeline is: expand replays -> swgy-spatial -> re-run cells.")
else:
    from swgy_tools.spatial.build import build as spatial_build
    from swgy_tools.tasks.expand import run_expand_replay

    replays = sorted(Path("replays").glob("*.replay")) or [Path("bundles") / "sample.replay"]
    exp = Path("expanded"); exp.mkdir(exist_ok=True)
    for r in replays:
        dest = exp / f"{r.stem}.jsonl"
        if not dest.exists():
            dest.write_text("\n".join(run_expand_replay(Path(EXPAND), r, snapshot_every=1)))
    print(f"expanded {len(replays)} replay(s) -> {exp}/")

    spatial_build([str(exp)], "mydata")
    my_kills = load_kills("mydata")
    my_sight = load_sightings("mydata")
    seen = sum(1 for k in my_kills
               if witnesses([s for s in my_sight if s["episode_id"] == k["episode_id"]], k))
    print(f"\nYOUR build: {len(my_kills)} kills, {seen} witnessed "
          f"({100*seen/max(len(my_kills),1):.0f}%)")
    print("Now re-run any cell above with SP = Path('mydata') -- same tables, your games.")


## 8 · The improvement loop

**play → pull → expand → build → analyze → change ONE thing → tune the knob offline (Let's
Play, knob-tuning section) → resubmit → measure again.** The scoreboard for “measure again” is
below; the scenarios above are the diagnosis in the middle.

What to hand your coding agent, per scenario:

| scenario | artifact (agent-native) | picture (multimodal) | made with |
|---|---|---|---|
| tasks | `tasks.csv`, `legs.csv`, `tour.csv` | `data/swimlanes.png` | `swgy-task-report`, `swgy-route`, `swgy-task-viz` |
| voting | the participation table + episode JSONL | — | this notebook’s scan of `expand_replay` output |
| dying | `kills.csv` + `visibility.csv` | `data/ribbon.png`, `data/killcam.png` | `swgy-spatial`, `swgy-kill-plot` |
| kill sites | witnessed-by-room table + `heatmaps.npz` | the tell / bodyhide maps | `swgy-spatial-render` |
| one bad game | `expand_replay game.replay > game.txt` — the full text timeline | — | `expand_replay` |

Every picture in this notebook has a CLI twin in that column, so the whole loop scripts
without the notebook. And when replay data can’t answer the question (a live protocol
detail, a doubt about what the server actually streams), `swgy-capture` records the raw
`sprite_v1` stream straight off a running game.


In [ ]:
# The scoreboard: side-win rate per policy over the bundle. crew members win when the
# outcome is "crew"; imposters when it is "imposter"; draws count against everyone.
assert "ROLES" in globals(), "run the voting scenario's scan cell first -- it builds ROLES/OUTCOMES"
wins = defaultdict(lambda: [0, 0])
for (ep, slot), role in ROLES.items():
    name = roster.get(ep, {}).get(str(slot))   # roster.json keys are strings (JSON objects)
    if name is None:
        continue
    outcome = OUTCOMES.get(ep, "unknown")
    wins[name][1] += 1
    if (role == "crew" and outcome == "crew") or (role == "imposter" and outcome == "imposter"):
        wins[name][0] += 1

print(f"{'policy':20} {'games':>6} {'side wins':>10} {'win rate':>9}")
for name, (w, n) in sorted(wins.items(), key=lambda kv: -(kv[1][0] / max(kv[1][1], 1))):
    print(f"{name:20} {n:6d} {w:10d} {100*w/n:8.0f}%")
print("\nThe +100 win dwarfs task points. Every scenario above is in service of this column.")


In [ ]:
# The bundle in four numbers -- the panel worth keeping. Every value here was computed by
# a cell above (nothing quoted from offline runs), so the same cell renders YOUR numbers
# once SP points at your own build.
import statistics as st

from swgy_tools.plotstyle import MUTED, PAGE, SECONDARY, SURFACE

needed = {"n_seen": "kill sites", "murdered_rate": "tasks (ghost economy)",
          "ts": "tasks (baselines)"}
missing = [f"{k}: run the {where} scenario first" for k, where in needed.items()
           if k not in globals()]
assert not missing, missing

kpis = [
    (f"{100 * n_seen / len(kills):.0f}%", "of kills had a third-party witness",
     "the engine's own line-of-sight log"),
    (f"{100 * wit_room['Bridge'] / tot_room['Bridge']:.0f}%", "witnessed rate in the Bridge",
     "where imposters kill most; the quiet bays run ~0%"),
    (f"{murdered_rate:.0f}% vs {survivor_rate:.0f}%", "assignment done: murdered vs survivors",
     "the ghost shift out-works the living"),
    (f"{st.median([t.ratio for t in ts]):.2f}x", "task order vs the optimal tour",
     "movement is already solved -- aim elsewhere"),
]
fig, axes = plt.subplots(2, 2, figsize=(11.5, 4.4), facecolor=PAGE)
fig.subplots_adjust(hspace=0.14, wspace=0.06, top=0.86)
for ax, (big, label, sub) in zip(axes.ravel(), kpis):
    ax.set_facecolor(SURFACE)
    ax.set_xticks([])
    ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.text(0.05, 0.56, big, color="white", fontsize=25, weight="bold", transform=ax.transAxes)
    ax.text(0.05, 0.36, label, color=SECONDARY, fontsize=11, transform=ax.transAxes)
    ax.text(0.05, 0.18, sub, color=MUTED, fontsize=9, transform=ax.transAxes)
fig.suptitle("30 league games, in numbers", color="white", fontsize=13, x=0.125, ha="left")
plt.show()
